# Classifying texts

The `Classification` module contains classes and methods that allows an user to perform classification on textual data by creating, training, and using machine learning models.

The module contains 2 core components:

1. `Classifier`: The core classifier class which dictates the flow of all machine learning models, always abiding the flow of preprocessing the data, initializing the desired model, training it, and evaluating the results
1. `Pipeline`: Abstract class from which all models must inherit from. Children class must define how a model will execute its trainign stage and how it will extract its own features

In this tutorial, we will understand how to use setup and create a Classifier instance to perform a classification experiment on real data. We will first load and process our data to then setup a specific `Pipeline` stragegy to be given to our `Classifier` instance

# Loading and Preparing Your Documents

Lets use the `Loader` class from Lexos to import our data contained in the sub-folder `fed_papers`, so we can then use `Tokenizer` and `Scrubber` to preprocess the data. We shall use Madison's and Hamilton's texts as the training data, and shall aim to predict the authorship of the contested and disputed papers 

In [1]:
from lexos.io.loader import Loader

loader = Loader()
loader.reset()  # Good practice in case there is previous loaded data

loader.load(
    "fed_papers"
)  # Assuming you are running this notebook from lexos/doc_src/docs/tutorials/classification

# Let's check for any errors and what files did we load
print("Errors:", loader.errors)
print("Loaded Names:", loader.names)

SEED = 42  # For reproducing this experiment

from lexos.scrubber.scrubber import Scrubber

scrubber = Scrubber()
scrubber.add_pipe(
    "lower_case"
)  # Simple scrubbing as to maintain feature extraction as ample as possible

# Now let's organize our data into training and test sets
df_all = loader.df
df_train = df_all[
    df_all["name"].str.contains(r"(?:_H|_M)$", case=False, regex=True, na=False)
].copy()
df_train["label"] = df_train["name"].apply(
    lambda name: "HAMILTON" if name.upper().endswith("_H") else "MADISON"
)
df_test = df_all[
    df_all["name"].str.contains(r"(?:_D|_C)$", case=False, regex=True, na=False)
].copy()

train_texts = df_train["text"].tolist()
train_labels = df_train["label"].tolist()
train_ids = df_train["name"].tolist()
test_texts = df_test["text"].tolist()
test_ids = df_test["name"].tolist()

# A final check to see our training and test data sets
print(f"Training docs extracted from Loader: {len(train_texts)}")
print(f"Test docs extracted from Loader: {len(test_texts)}")

Errors: []
Loaded Names: ['FED_18_C', 'FED_19_C', 'FED_20_C', 'FED_49_D', 'FED_50_D', 'FED_51_D', 'FED_52_D', 'FED_53_D', 'FED_54_D', 'FED_55_D', 'FED_56_D', 'FED_57_D', 'FED_58_D', 'FED_62_D', 'FED_63_D', 'FED_11_H', 'FED_12_H', 'FED_13_H', 'FED_15_H', 'FED_16_H', 'FED_17_H', 'FED_1_H', 'FED_21_H', 'FED_22_H', 'FED_23_H', 'FED_24_H', 'FED_25_H', 'FED_26_H', 'FED_27_H', 'FED_28_H', 'FED_29_H', 'FED_30_H', 'FED_31_H', 'FED_32_H', 'FED_33_H', 'FED_34_H', 'FED_35_H', 'FED_36_H', 'FED_59_H', 'FED_60_H', 'FED_61_H', 'FED_65_H', 'FED_66_H', 'FED_67_H', 'FED_68_H', 'FED_69_H', 'FED_6_H', 'FED_70_H', 'FED_71_H', 'FED_72_H', 'FED_73_H', 'FED_74_H', 'FED_75_H', 'FED_76_H', 'FED_77_H', 'FED_78_H', 'FED_79_H', 'FED_7_H', 'FED_80_H', 'FED_81_H', 'FED_82_H', 'FED_83_H', 'FED_84_H', 'FED_85_H', 'FED_8_H', 'FED_9_H', 'FED_2_J', 'FED_3_J', 'FED_4_J', 'FED_5_J', 'FED_64_J', 'FED_10_M', 'FED_14_M', 'FED_37_M', 'FED_38_M', 'FED_39_M', 'FED_40_M', 'FED_41_M', 'FED_42_M', 'FED_43_M', 'FED_44_M', 'FED_45_M',

# Creating a Pipeline

We shall now create an `Pipeline` configuration, which will dictate how our future `Classifier` should perform its internal steps of preprocessing data, initializing a model, training itself, and evaluating its results. For our experiment, we will create an `MLPPipeline` strategy, but this could be any other available Pipeline, or even one of your own!

In [ ]:
from lexos.classification.decision_tree_pipeline import DecisionTreePipeline

my_strategy = DecisionTreePipeline(
    seed=SEED,
    include_bigrams=True,
    tree_kwargs={
        "max_depth": 15, # Limits the maximum depth of our tree
        "min_samples_split": 10, # Sets the minimum num of document samples required to split an internal mode 
        "min_samples_leaf": 4, # Sets the minimum num of documents in each leaf node
        "max_features": "sqrt", # Limits the evaluated features at each split to the sqrt of the total vocab size
        "criterion": "entropy", # Selects entropy over Gini impurity for split quality
    },
)

Now that we have a `Pipeline` configured, let's pass it to a `Classifier` instance

In [3]:
from lexos.classification import Classifier

classifier = Classifier(
    train_data=train_texts, labels=train_labels, pipeline=my_strategy, features=None
)

Since our classifier now has an MLPStrategy to follow, we can fit it

In [4]:
classifier.fit()

TypeError: DecisionTreeClassifier.__init__() got an unexpected keyword argument 'max_feattures'

In [ ]:
classifier.metrics

{'accuracy': 0.9230769230769231,
 'balanced_accuracy': 0.95,
 'macro_f1': 0.9022556390977443}

In [ ]:
classifier.report

,precision,recall,f1-score,support
HAMILTON,1.000000,0.900000,0.947368,10.000000
MADISON,0.750000,1.000000,0.857143,3.000000
accuracy,0.923077,0.923077,0.923077,0.923077
macro avg,0.875000,0.950000,0.902256,13.000000
weighted avg,0.942308,0.923077,0.926547,13.000000


Now that we have trained our model, we can predict the test set with one simple call

In [ ]:
predictions_df = classifier.predict(
    data=train_texts, ids=train_ids, true_labels=train_labels
)

The `predict()` method orchestrates a systematic machine learning pipeline: first, it automatically filters and sorts the incoming DataFrame's columns to align perfectly with the `baseline_features` established during training, preventing shape or sorting errors; second, it applies the exact same `StandardScaler` used during training to ensure mathematical consistency; third, it passes those scaled values through the fitted Multi-Layer Perceptron neural network to output the classification targets; and finally, it binds those raw predictions side-by-side with your provided document IDs and true labels into a structured DataFrame for easy error analysis.

In [ ]:
import pandas as pd

predictions_df

,doc_id,true_label,predicted_label
0,FED_11_H,HAMILTON,HAMILTON
1,FED_12_H,HAMILTON,HAMILTON
2,FED_13_H,HAMILTON,HAMILTON
3,FED_15_H,HAMILTON,HAMILTON
4,FED_16_H,HAMILTON,HAMILTON
...,...,...,...
60,FED_44_M,MADISON,MADISON
61,FED_45_M,MADISON,MADISON
62,FED_46_M,MADISON,MADISON
63,FED_47_M,MADISON,MADISON


In [ ]:
# Filter and display only the rows where the model guessed wrong
mismatches_df = predictions_df[
    predictions_df["true_label"] != predictions_df["predicted_label"]
]

print(f"Total misclassifications: {len(mismatches_df)}")
mismatches_df

Total misclassifications: 0


,doc_id,true_label,predicted_label
